This notebook simply makes a test for the creation of reactions in ChEBI format, instead of names. This will be implemented in 1+2/1+2.ipynb.

In [31]:
import pandas as pd
import re
import numpy as np

First is the inital for the TCDB reactions\
Write this section below as a function as well, as done on the Rhea-part. Easier to integrate cleanly.

In [44]:
df = pd.read_csv("../1+2/transporters_df.tsv", sep="\t", dtype=str)

name2chebi_dict = {}

with open("name2chebi.txt", "r") as file:
    for line in file:
        parts = line.strip().split(" ")
        if len(parts) == 2:
            name, chebi_id = parts
            name2chebi_dict[name] = chebi_id

def tcdb_convert_to_chebi(row):
    reaction = row["Reaction"]
    
    if pd.notna(reaction) and isinstance(reaction, str):

        reaction = re.sub(r"\s*\(in\)|\(out\)", "", reaction)

        chebi_name = row["CHEBI Name"]
        chebi_id = row["CHEBI ID"]
        
        if pd.notna(chebi_name) and pd.notna(chebi_id):
            reaction = reaction.replace(chebi_name, chebi_id)

        for name, chebi_id in name2chebi_dict.items():
            reaction = reaction.replace(name, chebi_id)
        
    return reaction

df["TCDB:Reaction:CHEBI"] = df.apply(tcdb_convert_to_chebi, axis=1)
df

,AID,UID,TCID,CHEBI ID,CHEBI Name,AA,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier,TCDB:Reaction:CHEBI
0,A0CIB0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cl- (out) = Cl- (in),Cl-,chloride (out) = chloride (in),NaN,NaN,NaN,NaN,CHEBI:17996 = CHEBI:17996
1,A0CIB0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,cation (out) = cation (in),cation,chloride (out) = chloride (in),NaN,NaN,NaN,NaN,CHEBI:17996 = CHEBI:17996
2,A0CS82,A0CS82,9.B.82.1.5,NaN,NaN,MIIEEQIEEKMIYKAIHRVKVNYQKKIDRYILYKKSRWFFNLLLML...,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0CX44,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),MSQPITYSSLISLSLAKFPQVYMYTDGFMSNDFELISFNSVHGNLF...,1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,MINKAGKQLTWLFALKILSRIFDLSLNILVLRDLEPGIYGLTTNLD...,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57304,NP_001279001.1,A7M7C5,1.A.8.8.28,CHEBI:15377,water,MMWELRSVAFTRAVLAEFLATLVFILFGLGSALNWPSASAPSILQI...,1.A.8,solute (out) → solute (in),solute,water (out) → water (in),RHEA:29675,glycerol(in) = glycerol(out),glycerol,CHEBI:17754,CHEBI:15377 → CHEBI:15377
57305,NP_653300.2,Q13733,3.A.3.1.16,CHEBI:29101,sodium(1+),MGLWGKKGTVAPHDQSPRRRPKKGLIKKKMVKREKQKRNMEELKKE...,3.A.3,n Me1 (out) + m Me2 (in) + ATP → n Me1 (in) + ...,Me1,n sodium(1+) (out) + m Me2 (in) + ATP → n sodi...,RHEA:18353,K(+)(out) + Na(+)(in) + ATP + H2O = K(+)(in) +...,K(+);Na(+);ATP;H2O;ADP;phosphate;H(+),CHEBI:29103;CHEBI:29101;CHEBI:30616;CHEBI:1537...,n CHEBI:29101 + m CHEBI:33521 + CHEBI:15422 →...
57306,NP_653300.2,Q13733,3.A.3.1.16,CHEBI:29101,sodium(1+),MGLWGKKGTVAPHDQSPRRRPKKGLIKKKMVKREKQKRNMEELKKE...,3.A.3,Phospholipid (outer leaflet of the membrane) +...,Phospholipid,sodium(1+) (outer leaflet of the membrane) + A...,RHEA:18353,K(+)(out) + Na(+)(in) + ATP + H2O = K(+)(in) +...,K(+);Na(+);ATP;H2O;ADP;phosphate;H(+),CHEBI:29103;CHEBI:29101;CHEBI:30616;CHEBI:1537...,CHEBI:29101 (outer leaflet of the membrane) + ...
57307,NP_653300.2,Q13733,3.A.3.1.16,CHEBI:29103,potassium(1+),MGLWGKKGTVAPHDQSPRRRPKKGLIKKKMVKREKQKRNMEELKKE...,3.A.3,n Me1 (out) + m Me2 (in) + ATP → n Me1 (in) + ...,Me1,n potassium(1+) (out) + m Me2 (in) + ATP → n p...,RHEA:18353,K(+)(out) + Na(+)(in) + ATP + H2O = K(+)(in) +...,K(+);Na(+);ATP;H2O;ADP;phosphate;H(+),CHEBI:29103;CHEBI:29101;CHEBI:30616;CHEBI:1537...,n CHEBI:29103 + m CHEBI:33521 + CHEBI:15422 →...


The next step, is to create the reaction for Rhea as well, in ChEBI format. In R:ChEBI identifier, the ChEBIs appear in the order they appear in the reaction. That is handy.

In [45]:
def rhea_convert_to_chebi(row):
    equation = str(row["R:Equation"]) if pd.notnull(row["R:Equation"]) else ""
    chebi_str = str(row["R:ChEBI identifier"]) if pd.notnull(row["R:ChEBI identifier"]) else ""

    # Avoid errors when empty
    if not equation.strip() or not chebi_str.strip():
        return np.nan

    chebis = [c.strip() for c in chebi_str.split(";")]

    # Clean equation text
    cleaned = re.sub(r"\((in|out)\)", "", equation)
    cleaned = re.sub(r"\(n\+1\)", "", cleaned)
    cleaned = re.sub(r"\(n\)", "", cleaned)
    cleaned = re.sub(r"\s+n\s+", " ", cleaned)
    cleaned = re.sub(r"\s{2,}", " ", cleaned).strip()

    # Split on " + " and " = ", preserving operators
    tokens = re.split(r" ([+=]) ", cleaned)
    parts = [t.strip() for t in tokens if t.strip()]

    processed_names = []
    processed_ops = []

    i = 0
    while i < len(parts):
        token = parts[i]
        if token in ["+", "="]:
            processed_ops.append(f" {token} ")
        else:
            match = re.match(r"^(\d+)\s+(.+)$", token)
            if match:
                stoich, name = match.groups()
                processed_names.append((name, stoich))
            else:
                processed_names.append((token, ""))  # no stoichiometry
        i += 1

    # Map names to CHEBI IDs
    name_to_chebi = {}
    chebi_index = 0
    chebi_result = []
    
    for name, stoich in processed_names:
        if name not in name_to_chebi:
            if chebi_index < len(chebis):
                name_to_chebi[name] = chebis[chebi_index]
                chebi_index += 1
            else:
                name_to_chebi[name] = "MISSING_CHEBI" # Not an issue now, but kept in for later iterations
        chebi_id = name_to_chebi[name]
        chebi_result.append(f"{stoich} {chebi_id}".strip())

    # Reconstruct final reaction string
    result = [chebi_result[0]]
    for i, op in enumerate(processed_ops):
        result.append(op)
        result.append(chebi_result[i + 1])

    return "".join(result)

df["Rhea:Reaction:CHEBI"] = df.apply(rhea_convert_to_chebi, axis=1)
df

,AID,UID,TCID,CHEBI ID,CHEBI Name,AA,Family,Mechanism,Acting Entity,Reaction,RID,R:Equation,R:ChEBI name,R:ChEBI identifier,TCDB:Reaction:CHEBI,Rhea:Reaction:CHEBI
0,A0CIB0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,Cl- (out) = Cl- (in),Cl-,chloride (out) = chloride (in),NaN,NaN,NaN,NaN,CHEBI:17996 = CHEBI:17996,NaN
1,A0CIB0,A0CIB0,1.A.17.1.13,CHEBI:17996,chloride,MDDQNQPILQEQPKPKQKKPLLNTKMVKKQKMQNKKEENLREILNF...,1.A.17,cation (out) = cation (in),cation,chloride (out) = chloride (in),NaN,NaN,NaN,NaN,CHEBI:17996 = CHEBI:17996,NaN
2,A0CS82,A0CS82,9.B.82.1.5,NaN,NaN,MIIEEQIEEKMIYKAIHRVKVNYQKKIDRYILYKKSRWFFNLLLML...,9.B.82,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,A0CX44,A0CX44,1.A.3.2.4,CHEBI:29108,calcium(2+),MSQPITYSSLISLSLAKFPQVYMYTDGFMSNDFELISFNSVHGNLF...,1.A.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,A0D5K0,A0D5K0,2.A.66.3.4,CHEBI:8150,phospholipid,MINKAGKQLTWLFALKILSRIFDLSLNILVLRDLEPGIYGLTTNLD...,2.A.66,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
57304,NP_001279001.1,A7M7C5,1.A.8.8.28,CHEBI:15377,water,MMWELRSVAFTRAVLAEFLATLVFILFGLGSALNWPSASAPSILQI...,1.A.8,solute (out) → solute (in),solute,water (out) → water (in),RHEA:29675,glycerol(in) = glycerol(out),glycerol,CHEBI:17754,CHEBI:15377 → CHEBI:15377,CHEBI:17754 = CHEBI:17754
57305,NP_653300.2,Q13733,3.A.3.1.16,CHEBI:29101,sodium(1+),MGLWGKKGTVAPHDQSPRRRPKKGLIKKKMVKREKQKRNMEELKKE...,3.A.3,n Me1 (out) + m Me2 (in) + ATP → n Me1 (in) + ...,Me1,n sodium(1+) (out) + m Me2 (in) + ATP → n sodi...,RHEA:18353,K(+)(out) + Na(+)(in) + ATP + H2O = K(+)(in) +...,K(+);Na(+);ATP;H2O;ADP;phosphate;H(+),CHEBI:29103;CHEBI:29101;CHEBI:30616;CHEBI:1537...,n CHEBI:29101 + m CHEBI:33521 + CHEBI:15422 →...,CHEBI:29103 + CHEBI:29101 + CHEBI:30616 + CHEB...
57306,NP_653300.2,Q13733,3.A.3.1.16,CHEBI:29101,sodium(1+),MGLWGKKGTVAPHDQSPRRRPKKGLIKKKMVKREKQKRNMEELKKE...,3.A.3,Phospholipid (outer leaflet of the membrane) +...,Phospholipid,sodium(1+) (outer leaflet of the membrane) + A...,RHEA:18353,K(+)(out) + Na(+)(in) + ATP + H2O = K(+)(in) +...,K(+);Na(+);ATP;H2O;ADP;phosphate;H(+),CHEBI:29103;CHEBI:29101;CHEBI:30616;CHEBI:1537...,CHEBI:29101 (outer leaflet of the membrane) + ...,CHEBI:29103 + CHEBI:29101 + CHEBI:30616 + CHEB...
57307,NP_653300.2,Q13733,3.A.3.1.16,CHEBI:29103,potassium(1+),MGLWGKKGTVAPHDQSPRRRPKKGLIKKKMVKREKQKRNMEELKKE...,3.A.3,n Me1 (out) + m Me2 (in) + ATP → n Me1 (in) + ...,Me1,n potassium(1+) (out) + m Me2 (in) + ATP → n p...,RHEA:18353,K(+)(out) + Na(+)(in) + ATP + H2O = K(+)(in) +...,K(+);Na(+);ATP;H2O;ADP;phosphate;H(+),CHEBI:29103;CHEBI:29101;CHEBI:30616;CHEBI:1537...,n CHEBI:29103 + m CHEBI:33521 + CHEBI:15422 →...,CHEBI:29103 + CHEBI:29101 + CHEBI:30616 + CHEB...
